In [ ]:
# my
import pandas as pd
import numpy as np
from scipy.stats import norm
import os
import chardet
import warnings

warnings.filterwarnings('ignore')

# --- 1. 智慧掛載 (如果沒掛過才掛) ---
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# --- 2. 路徑設定 (修正為你的 HW2 路徑) ---
base_path = "/content/drive/MyDrive/Colab Notebooks/HW2"
data_folder = os.path.join(base_path, "Option_2021")  # 你已經解壓好的資料夾
output_file = os.path.join(base_path, "IV_Summary_Report.csv")

# --- 3. Black-Scholes 與 二分法計算 IV ---
def black_scholes_call(S, K, T, r, sigma):
    if sigma <= 0 or S <= 0 or K <= 0 or T <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def find_iv(market_price, S, K, T, r):
    if market_price <= 0: return 0
    low, high = 0.0001, 5.0
    for i in range(40):
        mid = (low + high) / 2
        price = black_scholes_call(S, K, T, r, mid)
        if price < market_price: low = mid
        else: high = mid
    return mid

# --- 4. 批次處理與清洗 (核心邏輯) ---
def calculate_iv_summary():
    if not os.path.exists(data_folder):
        print(f"❌ 找不到資料夾: {data_folder}")
        return

    all_iv_data = []
    file_list = sorted([f for f in os.listdir(data_folder) if f.endswith('.csv')])

    print(f"🚀 開始處理 {len(file_list)} 個檔案...")

    # 外部參數設定
    S_VALUE = 15000
    R_VALUE = 0.01
    T_DAYS = 20

    for f_name in file_list:
        f_path = os.path.join(data_folder, f_name)

        try:
            # 自動偵測編碼
            with open(f_path, 'rb') as f_raw:
                enc = chardet.detect(f_raw.read(10000))['encoding']
                if enc is None: enc = 'utf-8'

            data_rows = []
            with open(f_path, 'r', encoding=enc, errors='ignore') as f:
                for line in f:
                    # 洗掉橫線列與空列
                    if '---' in line or not line.strip():
                        continue

                    # 徹底清洗：用逗號切開並 strip
                    tokens = [t.strip() for t in line.split(',') if t.strip()]

                    # 根據你提供的截圖欄位 (0:日期, 2:履約, 4:買賣權, 6:成交價)
                    if len(tokens) >= 7 and tokens[4].upper() == 'C':
                        try:
                            data_rows.append([tokens[0], float(tokens[2]), float(tokens[6])])
                        except:
                            continue

            if not data_rows:
                continue

            # 建立單日 DataFrame
            df_day = pd.DataFrame(data_rows, columns=['Date', 'K', 'Price'])
            df_day['Date'] = pd.to_datetime(df_day['Date'], errors='coerce')

            # 計算 IV
            df_day['IV'] = df_day.apply(
                lambda x: find_iv(x['Price'], S_VALUE, x['K'], T_DAYS/365, R_VALUE), axis=1
            )

            # 只保留日期與 IV
            all_iv_data.append(df_day[['Date', 'IV']])
            print(f"✅ {f_name} 已完成")

        except Exception as e:
            print(f"❌ 檔案 {f_name} 出錯: {e}")

    if not all_iv_data:
        print("💀 沒有成功處理任何數據")
        return

    # 合併所有天數
    df_total = pd.concat(all_iv_data).dropna()

    # --- 5. 產出與範例圖一模一樣的統計格式 ---
    print("\n📊 正在生成統計報告...")

    # 使用 describe() 獲得 count, mean, std, min, 25%, 50%, 75%, max
    summary_df = df_total.groupby('Date')['IV'].describe()

    # 格式微調：count 轉整數
    summary_df['count'] = summary_df['count'].astype(int)

    # 格式化日期索引 (2021/1/4)
    summary_df.index = pd.to_datetime(summary_df.index).strftime('%Y/%-m/%-d')

    # 顯示結果
    display(summary_df)

    # 存檔
    summary_df.to_csv(output_file, encoding='utf-8-sig')
    print(f"\n✅ 完成！輸出位置: {output_file}")

# 執行
calculate_iv_summary()

🚀 開始處理 244 個檔案...


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import os
import chardet
import warnings

warnings.filterwarnings('ignore')

# --- 1. 智慧掛載 ---
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# --- 2. 路徑設定 (請根據你的雲端硬碟實際情況微調) ---
base_path = "/content/drive/MyDrive/Colab Notebooks/HW2"
# 假設你的索引檔放在 HW2 或是同學原本的路徑，請確認下面這行
index_file_path = os.path.join(base_path, "Path_教學_0409.csv")
output_summary_file = os.path.join(base_path, "IV_Final_Summary.csv")

# --- 3. 模型定義 ---
def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0: return max(0.0, S - K)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def find_iv(market_price, S, K, T, r):
    if market_price <= 0 or market_price < (S - K): return 0.00001
    low, high = 1e-5, 5.0
    for _ in range(40):
        mid = (low + high) / 2.0
        price = bs_call_price(S, K, T, r, mid)
        if price - market_price > 0: high = mid
        else: low = mid
    return mid

# --- 4. 主程式：讀取索引並處理 ---
def process_with_index():
    print("🔎 讀取索引檔...")
    try:
        # 抗亂碼讀取
        df_index = pd.read_csv(index_file_path, encoding='utf-8-sig')
    except:
        df_index = pd.read_csv(index_file_path, encoding='big5')

    df_index.columns = df_index.columns.str.strip()

    all_day_results = []

    for _, row in df_index.iterrows():
        f_name = str(row['File']).strip()
        f_path = os.path.join(base_path, f_name)
        s0 = float(row['S0'])
        rf = float(row['Rf'])
        target_contract = str(row['Contract']).strip()

        if not os.path.exists(f_path):
            print(f"⚠️ 找不到檔案: {f_name}，跳過。")
            continue

        print(f"🚀 處理中: {f_name} (S0: {s0}, 合約: {target_contract})")

        # 💡 核心改動：不再用 read_csv，改用暴力拆解防止 Expected Fields 錯誤
        day_data = []
        try:
            with open(f_path, 'r', encoding='utf-8', errors='ignore') as f:
                lines = f.readlines()
        except:
            with open(f_path, 'r', encoding='big5', errors='ignore') as f:
                lines = f.readlines()

        for line in lines:
            if '---' in line or not line.strip() or '成交日期' in line:
                continue

            # 認真清洗：切分後 strip 每個元素
            tokens = [t.strip() for t in line.split(',') if t.strip()]

            # 欄位定義依據截圖：
            # 0:日期, 1:商品(TXO), 2:履約, 3:月份(Contract), 4:買賣別, 6:成交價, 7:成交量
            if len(tokens) >= 8:
                try:
                    product = tokens[1].upper()
                    strike = float(tokens[2])
                    contract = tokens[3].strip()
                    cp_flag = tokens[4].upper()
                    price = float(tokens[6])
                    volume = float(tokens[7])

                    # 篩選：TXO + 指定月份 + 買權 + 成交量 > 30
                    if product == 'TXO' and contract == target_contract and cp_flag == 'C' and volume > 30:
                        day_data.append([tokens[0], strike, price])
                except:
                    continue

        if day_data:
            df_day = pd.DataFrame(day_data, columns=['Date', 'K', 'Price'])
            # 計算 IV
            T = 20.0 / 365.0 # 固定天數或可根據索引檔調整
            df_day['IV'] = df_day.apply(lambda x: find_iv(x['Price'], s0, x['K'], T, rf), axis=1)
            all_day_results.append(df_day[['Date', 'IV']])
            print(f"   -> 成功提取 {len(df_day)} 筆資料")

    # --- 5. 整合產出統計表 ---
    if all_day_results:
        final_df = pd.concat(all_day_results)
        final_df['Date'] = pd.to_datetime(final_df['Date'], errors='coerce')

        # 產出與你第一張截圖一模一樣的 8 個統計指標
        summary = final_df.groupby('Date')['IV'].describe()

        # 格式微調
        summary['count'] = summary['count'].astype(int)
        summary.index = summary.index.strftime('%Y/%-m/%-d')

        print("\n" + "="*60)
        display(summary)
        print("="*60)

        summary.to_csv(output_summary_file, encoding='utf-8-sig')
        print(f"🎉 任務完成！統計報表已儲存：{output_summary_file}")
    else:
        print("💀 失敗：沒有任何符合條件的資料被處理。")

# 執行
process_with_index()

🔎 讀取索引檔...
🚀 處理中: OptionsDaily_2020_01_02.csv (S0: 12100.48, 合約: 202001)
   -> 成功提取 279 筆資料
🚀 處理中: OptionsDaily_2020_01_03.csv (S0: 12110.43, 合約: 202001)
   -> 成功提取 527 筆資料
🚀 處理中: OptionsDaily_2020_01_06.csv (S0: 11953.36, 合約: 202001)
   -> 成功提取 492 筆資料
🚀 處理中: OptionsDaily_2020_01_07.csv (S0: 11880.32, 合約: 202001)
   -> 成功提取 1046 筆資料
🚀 處理中: OptionsDaily_2020_01_08.csv (S0: 11817.1, 合約: 202001)
   -> 成功提取 1449 筆資料



,count,mean,std,min,25%,50%,75%,max
Date,,,,,,,,
2019/12/31,41,0.096875,0.010478,0.058316,0.094784,0.100081,0.104671,0.104996
2020/1/2,256,0.111022,0.008869,0.059166,0.109782,0.112927,0.115586,0.127028
2020/1/3,554,0.106193,0.018114,0.000010,0.094709,0.100106,0.122485,0.171273
2020/1/4,3,0.124144,0.005398,0.121028,0.121028,0.121028,0.125703,0.130377
2020/1/6,547,0.117121,0.022744,0.099894,0.109354,0.114082,0.118451,0.358463
2020/1/7,1035,0.102814,0.011083,0.078147,0.094905,0.100950,0.109080,0.154159
2020/1/8,1357,0.091248,0.011823,0.066159,0.083827,0.088890,0.097735,0.178122


🎉 任務完成！統計報表已儲存：/content/drive/MyDrive/Colab Notebooks/HW2/IV_Final_Summary.csv


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import os
import chardet
import warnings

warnings.filterwarnings('ignore')

# --- 1. 智慧掛載 ---
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# --- 2. 路徑設定 (目標：2021 整年度資料夾) ---
base_path = "/content/drive/MyDrive/Colab Notebooks/HW2"
data_folder = os.path.join(base_path, "Option_2021") # 指向 2021 資料夾
output_file = os.path.join(base_path, "IV_Summary_2021_FullYear.csv")

# --- 3. 模型定義：Black-Scholes 與 二分法 ---
def black_scholes_call(S, K, T, r, sigma):
    if sigma <= 0 or S <= 0 or K <= 0 or T <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def find_iv(market_price, S, K, T, r):
    if market_price <= 0: return 0
    low, high = 0.0001, 5.0
    for i in range(40):
        mid = (low + high) / 2
        price = black_scholes_call(S, K, T, r, mid)
        if price < market_price: low = mid
        else: high = mid
    return mid

# --- 4. 核心批次處理邏輯 ---
def process_2021_all_files():
    if not os.path.exists(data_folder):
        print(f"❌ 找不到資料夾: {data_folder}")
        return

    # 獲取該資料夾下所有 CSV 並排序 (確保 244 個檔案按日期順序處理)
    file_list = sorted([f for f in os.listdir(data_folder) if f.endswith('.csv')])

    all_iv_data = []
    print(f"🚀 偵測到 {len(file_list)} 個檔案，開始處理 2021 全年度數據...")
    print(f"篩選條件：買權 (C) 且 成交量 > 30\n")

    # 💡 外部參數預設 (若你有索引檔也可改為動態讀取)
    S_VALUE = 15000
    R_VALUE = 0.01
    T_DAYS = 20

    for f_name in file_list:
        f_path = os.path.join(data_folder, f_name)

        try:
            # 偵測編碼
            with open(f_path, 'rb') as f_raw:
                enc_res = chardet.detect(f_raw.read(10000))
                enc = enc_res['encoding'] if enc_res['encoding'] else 'utf-8'

            data_rows = []
            # 認真清洗：逐行讀取
            with open(f_path, 'r', encoding=enc, errors='ignore') as f:
                for line in f:
                    # 移除橫線、空行、標題列
                    if '---' in line or not line.strip() or '成交日期' in line:
                        continue

                    # 核心清洗：使用逗號切分並 strip 每個元素
                    tokens = [t.strip() for t in line.split(',') if t.strip()]

                    # 欄位定義：0:日期, 2:履約, 4:買賣別, 6:成交價, 7:成交量
                    if len(tokens) >= 8:
                        try:
                            date_val = tokens[0]
                            strike_val = float(tokens[2])
                            cp_flag = tokens[4].upper()
                            price_val = float(tokens[6])
                            volume_val = float(tokens[7])

                            # 判定條件：買權 且 交易量 > 30
                            if cp_flag == 'C' and volume_val > 30:
                                data_rows.append([date_val, strike_val, price_val])
                        except:
                            continue

            if not data_rows:
                continue

            # 建立單日 DataFrame
            df_day = pd.DataFrame(data_rows, columns=['Date', 'K', 'Price'])
            df_day['Date'] = pd.to_datetime(df_day['Date'], errors='coerce')

            # 計算 IV
            df_day['IV'] = df_day.apply(
                lambda x: find_iv(x['Price'], S_VALUE, x['K'], T_DAYS/365, R_VALUE), axis=1
            )

            # 只保留日期與 IV 以節省記憶體
            all_iv_data.append(df_day[['Date', 'IV']])
            print(f"✅ {f_name} 處理成功")

        except Exception as e:
            print(f"❌ 處理檔案 {f_name} 時發生錯誤: {e}")

    # --- 5. 整合與生成 8 欄位統計表 ---
    if all_iv_data:
        final_df = pd.concat(all_iv_data).dropna()

        # 產出統計摘要 (count, mean, std, min, 25%, 50%, 75%, max)
        summary = final_df.groupby('Date')['IV'].describe()

        # 格式微調：count 轉整數
        summary['count'] = summary['count'].astype(int)

        # 格式化日期索引 (例如 2021/1/4)
        summary.index = pd.to_datetime(summary.index).strftime('%Y/%-m/%-d')

        print("\n" + "="*60)
        print("📊 2021 全年度 IV 統計報告：")
        display(summary)
        print("="*60)

        # 輸出 CSV
        summary.to_csv(output_file, encoding='utf-8-sig')
        print(f"🎉 報告已儲存至：{output_file}")
    else:
        print("\n💀 未找到符合條件的資料。")

# 啟動處理
process_2021_all_files()

🚀 偵測到 244 個檔案，開始處理 2021 全年度數據...
篩選條件：買權 (C) 且 成交量 > 30

✅ o20210104.csv 處理成功
✅ o20210105.csv 處理成功
✅ o20210106.csv 處理成功
✅ o20210107.csv 處理成功
✅ o20210108.csv 處理成功
✅ o20210111.csv 處理成功
✅ o20210112.csv 處理成功
✅ o20210113.csv 處理成功
✅ o20210114.csv 處理成功
✅ o20210115.csv 處理成功
✅ o20210118.csv 處理成功
✅ o20210119.csv 處理成功
✅ o20210120.csv 處理成功
✅ o20210121.csv 處理成功
✅ o20210122.csv 處理成功
✅ o20210125.csv 處理成功
✅ o20210126.csv 處理成功
✅ o20210127.csv 處理成功
✅ o20210128.csv 處理成功
✅ o20210129.csv 處理成功
✅ o20210201.csv 處理成功
✅ o20210202.csv 處理成功
✅ o20210203.csv 處理成功
✅ o20210204.csv 處理成功
✅ o20210205.csv 處理成功
✅ o20210217.csv 處理成功
✅ o20210218.csv 處理成功
✅ o20210219.csv 處理成功
✅ o20210222.csv 處理成功
✅ o20210223.csv 處理成功
✅ o20210224.csv 處理成功
✅ o20210225.csv 處理成功
✅ o20210226.csv 處理成功
✅ o20210302.csv 處理成功
✅ o20210303.csv 處理成功
✅ o20210304.csv 處理成功
✅ o20210305.csv 處理成功
✅ o20210308.csv 處理成功
✅ o20210309.csv 處理成功
✅ o20210310.csv 處理成功
✅ o20210311.csv 處理成功
✅ o20210312.csv 處理成功
✅ o20210315.csv 處理成功
✅ o20210316.csv 處理成功
✅ o20210317.csv 處理成

,count,mean,std,min,25%,50%,75%,max
Date,,,,,,,,
2020/12/31,47,0.032035,0.045202,0.000100,0.000100,0.000100,0.088800,0.107594
2021/1/1,17,0.006901,0.008961,0.000100,0.000100,0.005108,0.007411,0.025877
2021/1/4,1856,0.036118,0.041752,0.000100,0.000100,0.027454,0.043461,0.195116
2021/1/5,2059,0.039347,0.040689,0.000100,0.015455,0.032687,0.043307,0.212739
2021/1/6,4926,0.049336,0.049508,0.000100,0.014733,0.039267,0.059798,0.429612
...,...,...,...,...,...,...,...,...
2021/12/25,3,0.436557,0.106457,0.356120,0.376197,0.396274,0.476775,0.557276
2021/12/27,754,0.478736,0.097580,0.323843,0.414797,0.456039,0.532544,1.085741
2021/12/28,1585,0.484624,0.105309,0.000100,0.419063,0.463317,0.550820,1.096569


🎉 報告已儲存至：/content/drive/MyDrive/Colab Notebooks/HW2/IV_Summary_2021_FullYear.csv
